## Scalable path (`tensor_network_updates`)

The original `pstrings_to_mpo_optimized` builds the *exact* MPO (bond dimension
= number of terms `K`) via dense `(2,2,K,K)` diagonal tensors before truncating,
so it needs `O(K**2)` memory and dies for large `K`.

`tensor_network_updates` instead does a single left-to-right compression sweep
that never forms the full `K`-bond: memory is `O(max_bond * K)` and time is
`O(n_qubits * K * max_bond**2)`. This scales to ~40 qubits and
hundreds-of-thousands to millions of terms.

In [1]:
from symmer.operators import PauliwordOp
from symmer.approximate.tensor_network_updates import (
    mpo_from_pauliwordop,
    find_groundstate_quimb,
)
import numpy as np

# large qubit count + many terms
Hq = PauliwordOp.random(n_qubits=40, n_terms=200_000, complex_coeffs=False, density=0.6)

# PauliwordOp -> compressed MPO (bounded bond dimension, complex64 to save memory)
MPO = mpo_from_pauliwordop(Hq, max_bond=100, cutoff=1e-10, dtype=np.complex64)
MPO

/Users/lex/anaconda3/envs/boehringer/lib/python3.13/site-packages/symmer/approximate/tensor_network.py:7: FutureWarning: The module 'quimb.tensor.tensor_1d' is deprecated and will be removed in a future release. Most functionality can be still be accessed directly from 'quimb.tensor' instead. The actual implementations have moved to `quimb.tensor.tn1d.core`.
  from quimb.tensor.tensor_1d import MatrixProductOperator, MatrixProductState
/Users/lex/anaconda3/envs/boehringer/lib/python3.13/site-packages/symmer/approximate/tensor_network.py:8: FutureWarning: The module 'quimb.tensor.tensor_dmrg' is deprecated and will be removed in a future release. Most functionality can be still be accessed directly from 'quimb.tensor' instead. The actual implementations have moved to `quimb.tensor.tn1d.dmrg`.
  from quimb.tensor.tensor_dmrg import DMRG2


MatrixProductOperator(tensors=40, indices=119, L=40, max_bond=100)

In [2]:
# solve for the ground-state energy with DMRG.
# For 40 qubits do NOT densify the state (2**40 amplitudes); keep the MPS.
result = find_groundstate_quimb(
    MPO,
    bond_dims=[20, 50, 100, 100],
    cutoffs=1e-8,
    max_sweeps=8,
    verbosity=1,
)
print("ground-state energy:", result["energy"])

1, R, max_bond=(20/20), cutoff:1e-08


100%|###########################################| 39/39 [00:02<00:00, 16.91it/s]

Energy: (-2.7179763317108154-2.9802322387695312e-08j) ... not converged.
2, R, max_bond=(20/50), cutoff:1e-08



100%|###########################################| 39/39 [00:01<00:00, 31.26it/s]

Energy: (-12.58491325378418+2.086162567138672e-07j) ... not converged.
3, R, max_bond=(40/100), cutoff:1e-08


...

Energy: (-17.24784278869629+4.76837158203125e-07j) ... not converged.
7, R, max_bond=(100/100), cutoff:1e-08



100%|###########################################| 39/39 [00:04<00:00,  8.50it/s]

Energy: (-17.30499839782715+0j) ... not converged.
8, R, max_bond=(100/100), cutoff:1e-08



100%|###########################################| 39/39 [00:04<00:00,  9.59it/s]

Energy: (-17.360610961914062+0j) ... not converged.
ground-state energy: -17.360610961914062


In [3]:
result["mps"]

MatrixProductState(tensors=40, indices=79, L=40, max_bond=100)